In [11]:
import torch
import torch.nn as nn
import os
import pandas as pd
import joblib

In [12]:
encoder = joblib.load(r"C:\SML_Projects\SML_gym_fatPercentage_predict_project\model\encoder.joblib")
scaler = joblib.load(r"C:\SML_Projects\SML_gym_fatPercentage_predict_project\model\scaler.joblib")

c:\SML_Projects\SML_gym_fatPercentage_predict_project\env\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OrdinalEncoder from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\SML_Projects\SML_gym_fatPercentage_predict_project\env\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [13]:
class StructuredNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

In [14]:
model = StructuredNN(input_dim=17)

model.load_state_dict(torch.load(
    r"C:\SML_Projects\SML_gym_fatPercentage_predict_project\model\structured_nn.pth"
))

model.eval()

StructuredNN(
  (net): Sequential(
    (0): Linear(in_features=17, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): ReLU()
    (10): Linear(in_features=64, out_features=1, bias=True)
  )
)

# User data

In [15]:
user = pd.DataFrame({
    'age': [56],
    'gender': ['Male'],
    'weight': [88.3],
    'height': [1.71],
    'max_bpm': [180],
    'avg_bpm': [157],
    'resting_bpm': [60],
    'session_duration': [1.69],
    'calories_burned': [1313.0],
    'workout_type': ['Yoga'],
    'water_intake': [2.5],
    'workout_frequency': [4],
    'experience_level': [3],
    'bmi': [30.2],
    'age_category': ['50-100'],
    'weight_category': ['80-99'],
    'bmi_category': ['obesity']
})

In [16]:
user_raw = user.copy()

# ========= ENCODING =========
cat_cols = user.select_dtypes(include=['object']).columns

if len(cat_cols) > 0:
    user[cat_cols] = encoder.transform(user[cat_cols])

# ========= SCALING =========
user = scaler.transform(user)

# ========= TORCH =========
X = torch.tensor(user, dtype=torch.float32)

In [17]:
X = torch.tensor(user, dtype=torch.float32)

In [18]:
with torch.no_grad():
    pred = model(X).item()

In [19]:
pred = float(pred)

# Fat category logic
if user_raw['gender'][0].lower() == 'male':
    if pred < 6:
        category = "🩻 Essential Fat"
        meaning = "Organlar uchun minimal zarur yog‘"
    elif pred < 14:
        category = "🏆 Athlete Level"
        meaning = "Sportchi darajasi"
    elif pred < 18:
        category = "💪 Fitness Level"
        meaning = "Jismoniy tayyorgarlik yaxshi"
    elif pred < 25:
        category = "✅ Healthy Normal"
        meaning = "Sog‘lom va balansli"
    elif pred < 30:
        category = "⚠️ Overfat"
        meaning = "Yog‘ miqdori oshgan"
    else:
        category = "🚨 Obese Risk"
        meaning = "Sog‘liq uchun xavfli zona"
else:  # female
    if pred < 14:
        category = "🩻 Essential Fat"
        meaning = "Organlar uchun minimal zarur yog‘"
    elif pred < 21:
        category = "🏆 Athlete Level"
        meaning = "Sportchi darajasi"
    elif pred < 25:
        category = "💪 Fitness Level"
        meaning = "Jismoniy tayyorgarlik yaxshi"
    elif pred < 32:
        category = "✅ Healthy Normal"
        meaning = "Sog‘lom va balansli"
    elif pred < 36:
        category = "⚠️ Overfat"
        meaning = "Yog‘ miqdori oshgan"
    else:
        category = "🚨 Obese Risk"
        meaning = "Sog‘liq uchun xavfli zona"

print("\n🧬 BODY FAT ANALYSIS REPORT")
print("════════════════════════════")
print(f"👤 Gender        : {user_raw['gender'][0]}")
print(f"🏋️ Workout Type : {user_raw['workout_type'][0]}")
print(f"💧 Water Intake : {user_raw['water_intake'][0]} L")
print("────────────────────────────")
print(f"📊 Fat % Predict : {pred:.1f}%")
print(f"🧠 Category      : {category}")
print(f"📝 Meaning       : {meaning}")
print("────────────────────────────")

# Simple advice logic
if pred < 18:
    advice = "🥗 Ovqatlanish balansini oshirish + mushak massasi ko‘paytirish"
elif 18 <= pred < 25:
    advice = "✅ Hozirgi rejimni saqlash + barqaror trening"
elif 25 <= pred < 30:
    advice = "🔥 Cardio + HIIT ko‘paytirish + suv ichishni oshirish"
else:
    advice = "🚨 Nutrition plan + qat’iy trening rejasi + monitoring"

print(f"🎯 Recommendation : {advice}")
print("════════════════════════════\n")


🧬 BODY FAT ANALYSIS REPORT
════════════════════════════
👤 Gender        : Male
🏋️ Workout Type : Yoga
💧 Water Intake : 2.5 L
────────────────────────────
📊 Fat % Predict : 12.0%
🧠 Category      : 🏆 Athlete Level
📝 Meaning       : Sportchi darajasi
────────────────────────────
🎯 Recommendation : 🥗 Ovqatlanish balansini oshirish + mushak massasi ko‘paytirish
════════════════════════════

